Test Filings

In [9]:
from edgar import *
set_identity("dattasohamai@gmail.com")

In [10]:
class FilingFetcher:
    def __init__(self):
        set_identity("dev@gmail.com")


    def fetch_filings(
        self,
        ticker: str,
        start_year: int,
        form_types: list[str] = ['10-K', '10-Q', '8-K'],
        end_year: int | None = None,
    ):
        """
        Fetch financial SEC Edgar filings within a date range.
        """

        company = Company(ticker)

        start = f"{start_year}-01-01"
        end = f"{end_year}-12-31" if end_year else ""

        date_range = f"{start}:{end}"

        filings = company.get_filings(
            form=form_types,
            date=date_range
        )

        return filings

In [11]:
fetchdata = FilingFetcher()

In [12]:
filings = fetchdata.fetch_filings('AAPL', start_year=2025, end_year=2026,)

In [13]:
xbrl_data = filings[0].xbrl()
income_statement = xbrl_data.statements.income_statement().to_dataframe()
markdown_income_statememt = income_statement.to_markdown()
print(markdown_income_statememt)

|    | concept                                                                                             | label                                        | standard_concept               |   2026-03-28 (Q2) |   2025-03-29 (Q2) |   2026-03-28 (YTD) |   2025-03-29 (YTD) |   level | abstract   | dimension   | is_breakdown   | dimension_axis                        | dimension_member                       | dimension_member_label                  | dimension_label                                                                                             | balance   |   weight |   preferred_sign | parent_concept                                                                                      | parent_abstract_concept                                  |
|---:|:----------------------------------------------------------------------------------------------------|:---------------------------------------------|:-------------------------------|------------------:|------------------:|-----------

In [14]:
from langchain_core.documents import Document


def filings_to_langchain_docs(filings, ticker):
    """Convert filings into LangChain Document objects"""

    all_docs = []  # global accumulator

    for filing in filings:
        try:
            obj = filing.obj()
            chunk_doc = obj.chunked_document
            items = chunk_doc.list_items()

            for item in items:
                item_chunks = chunk_doc.chunks_for_item(item)  # local variable

                for i, c in enumerate(item_chunks):
                    if isinstance(c, str):
                        text = c
                    else:
                        text = getattr(c, "text", str(c))

                    # filtering
                    if not text or len(text.strip()) < 40:
                        continue
                    if "TableBlock" in text:
                        continue

                    all_docs.append(
                        Document(
                            page_content=text,
                            metadata={
                                "ticker": ticker,
                                "form": filing.form,
                                "filing_date": str(filing.filing_date),
                                "filing_year": filing.filing_date.year,
                                "section": item,
                                "chunk_id": i,
                                "source": "SEC"
                            }
                        )
                    )

        except Exception as e:
            print(f"Skipping filing {getattr(filing, 'form', 'unknown')} {getattr(filing, 'filing_date', 'unknown')}: {type(e).__name__}: {e}")
            continue

    return all_docs

In [15]:
docs = filings_to_langchain_docs(filings, 'AAPL')

In [16]:
docs[0].metadata

{'ticker': 'AAPL',
 'form': '10-Q',
 'filing_date': '2026-05-01',
 'filing_year': 2026,
 'section': 'Item 1',
 'chunk_id': 0,
 'source': 'SEC'}

In [17]:
company = Company("AAPL")
filings = company.get_filings()

In [3]:
type(filings)

edgar.entity.filings.EntityFilings

In [4]:
print(f"CIK value: {filings.cik}")
print(f"Company Name: {filings.company_name}")

CIK value: 320193
Company Name: Apple Inc.


In [5]:
#get latest filing
latest= filings.latest()
latest

╭───────────────────── Form 4 Apple Inc. [320193]  ─────────────────────╮
│                                                                       │
│   Accession Number       Filing Date   Period of Report   Documents   │
│  ───────────────────────────────────────────────────────────────────  │
│   0001140361-26-013192   2026-04-03    2026-04-01         1           │
│                                                                       │
╰─ Statement of changes in beneficial ownership • filing.docs for usage─╯

In [6]:
#filtering data collection
annual = filings.filter(form="10-k")
recent = filings.filter(filing_date="2026-01-01:")       
recent


╭──────────────────────────────────────── Filings for Apple Inc. [320193] ────────────────────────────────────────╮
│                                                                                                                 │
│                                                                              Filing                             │
│    Form        Description                                                   Date         Accession Number      │
│  ─────────────────────────────────────────────────────────────────────────────────────────────────────────────  │
│    4           Statement of changes in beneficial ownership                  2026-04-03   0001140361-26-0131…   │
│    4           Statement of changes in beneficial ownership                  2026-04-03   0001140361-26-0131…   │
│    4           Statement of changes in beneficial ownership                  2026-04-03   0001140361-26-0131…   │
│    144         Notice of proposed sale                                

In [7]:
#multiple recent filings
latest_5 = filings.latest(5)
latest_5

╭──────────────────────────────────────── Filings for Apple Inc. [320193] ────────────────────────────────────────╮
│                                                                                                                 │
│                                                                              Filing                             │
│    Form        Description                                                   Date         Accession Number      │
│  ─────────────────────────────────────────────────────────────────────────────────────────────────────────────  │
│    4           Statement of changes in beneficial ownership                  2026-04-03   0001140361-26-0131…   │
│    4           Statement of changes in beneficial ownership                  2026-04-03   0001140361-26-0131…   │
│    4           Statement of changes in beneficial ownership                  2026-04-03   0001140361-26-0131…   │
│    144         Notice of proposed sale                                

In [8]:
#Filter the collection

#by form type
annual = filings.filter(form="10K")
quarterly = filings.filter(form="10Q")
multiple = filings.filter(form=["10-K", "10-Q"])                                                              
multiple

╭──────────────────────────────────────── Filings for Apple Inc. [320193] ────────────────────────────────────────╮
│                                                                                                                 │
│                                                                              Filing                             │
│    Form        Description                                                   Date         Accession Number      │
│  ─────────────────────────────────────────────────────────────────────────────────────────────────────────────  │
│    10-Q        Quarterly report for public companies                         2026-01-30   0000320193-26-0000…   │
│    10-K        Annual report for public companies                            2025-10-31   0000320193-25-0000…   │
│    10-Q        Quarterly report for public companies                         2025-08-01   0000320193-25-0000…   │
│    10-Q        Quarterly report for public companies                  

In [ ]:
multiple = filings.filter(form=["10-K", "10-Q"])                                                              

In [9]:
recent_10K = filings.filter(form="10-K")[0]
print(recent_10K.text())

UNITED STATES

SECURITIES AND EXCHANGE COMMISSION

Washington, D.C. 20549

FORM 10-K

(Mark One)

☒ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934

For the fiscal year ended September 27, 2025

or

☐TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934

For the transition period from to.

Commission File Number: 001-36743

Apple Inc.

(Exact name of Registrant as specified in its charter)

  California                                    94-2404110                            
  (State or other jurisdiction                  (I.R.S. Employer Identification No.)  
  One Apple Park Way                                                                  
  Cupertino, California                         95014                                 
  (Address of principal executive offices)      (Zip Code)                            

( 408 996-1010

(Registrant’s telephone number, including area code)

Securities registered pursuan

In [ ]:
#converting to DataFrame
#all columns
df = filings.to_pandas()
df

In [48]:
#getting the most recernt filings
latest = filings.latest(5)
latest

╭──────────────────────────────────────── Filings for Apple Inc. [320193] ────────────────────────────────────────╮
│                                                                                                                 │
│                                                                              Filing                             │
│    Form        Description                                                   Date         Accession Number      │
│  ─────────────────────────────────────────────────────────────────────────────────────────────────────────────  │
│    4           Statement of changes in beneficial ownership                  2026-04-03   0001140361-26-0131…   │
│    4           Statement of changes in beneficial ownership                  2026-04-03   0001140361-26-0131…   │
│    4           Statement of changes in beneficial ownership                  2026-04-03   0001140361-26-0131…   │
│    144         Notice of proposed sale                                

In [49]:
#convert filings to dictiionary
dict_data = filings.to_dict()
dict_data

[{'accession_number': '0001140361-26-013192',
  'filing_date': datetime.date(2026, 4, 3),
  'reportDate': '2026-04-01',
  'acceptanceDateTime': Timestamp('2026-04-03 22:30:45+0000', tz='UTC'),
  'act': '',
  'form': '4',
  'fileNumber': '',
  'items': '',
  'size': 17348,
  'isXBRL': 0,
  'isInlineXBRL': 0,
  'primaryDocument': 'xslF345X06/form4.xml',
  'primaryDocDescription': 'FORM 4'},
 {'accession_number': '0001140361-26-013191',
  'filing_date': datetime.date(2026, 4, 3),
  'reportDate': '2026-04-01',
  'acceptanceDateTime': Timestamp('2026-04-03 22:30:43+0000', tz='UTC'),
  'act': '',
  'form': '4',
  'fileNumber': '',
  'items': '',
  'size': 14156,
  'isXBRL': 0,
  'isInlineXBRL': 0,
  'primaryDocument': 'xslF345X06/form4.xml',
  'primaryDocDescription': 'FORM 4'},
 {'accession_number': '0001140361-26-013190',
  'filing_date': datetime.date(2026, 4, 3),
  'reportDate': '2026-04-01',
  'acceptanceDateTime': Timestamp('2026-04-03 22:30:41+0000', tz='UTC'),
  'act': '',
  'form': 

In [52]:
print(company.text())

COMPANY: Apple Inc.
CIK: 0000320193
Ticker: AAPL
Exchange: Nasdaq
Industry: Electronic Computers (SIC 3571)
Entity Type: Operating
Category: Large accelerated filer
Fiscal Year End: Sep 26

AVAILABLE ACTIONS:
  - Use .get_filings() to access SEC filings
  - Use .financials to get financial statements
  - Use .facts to access company facts API
  - Use .docs for detailed API documentation


C:\Users\soham_ai\AppData\Local\Temp\ipykernel_19716\3962437349.py:1: DeprecationWarning: Company.text() is deprecated and will be removed in a future version. Use Company.to_context() instead for consistent naming.
  print(company.text())


In [62]:
type(company)

edgar.entity.core.Company

In [63]:
financials = company.get_financials()

In [73]:
financials = company.get_financials()

income = financials.income_statement()
balance = financials.balance_sheet()
cashflow = financials.cashflow_statement()
equity = financials.statement_of_equity()
comprehensive = financials.comprehensive_income()

In [87]:
equity

                                                                                                                   
                                         APPLE INC.   AAPL                                                         
                                         CONSOLIDATED STATEMENT OF EQUITY                                          
                                         Sep 30, 2023 to Sep 27, 2025                                              
                                                                                                                   
                                                                     Sep 27, 2025   Sep 28, 2024   Sep 30, 2023    
   ─────────────────────────────────────────────────────────────────────────────────────────────────────────────   
        Increase (Decrease) in Stockholders' Equity [Roll Forward]                                                 
            Ending balances - Beginning balance:                        